# 0. Подготовка проекта

In [34]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42
DATA_PATH = Path('data/Dataset.xlsx')

print(sys.version)
print('pandas:',pd.__version__)
print('scikit-learn:',sklearn.__version__)

3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
pandas: 3.0.6
scikit-learn: 1.9.1


# 1. Постановка задачи

В рассматриваемом датасете каждый объект наблюдения представляет собой попытку аутентификации. Необходимо определить, является ли конкретная попытка входа подозрительным событием в системе. Результат оценки используется специалистом по информационной безопасности, который на его основе может принять соответствующие меры реагирования, например дополнительно проверить попытку входа или ограничить доступ.

## 1.1. Объект анализа

Объект анализа – **одна попытка аутентификации**

Чтобы это доказать, выведем сколько раз встречается каждый пользователь.

In [35]:
df = pd.read_excel(DATA_PATH, sheet_name="Data")
df["user_id"].value_counts()

user_id
usr_062    24
usr_025    21
usr_082    19
usr_060    17
usr_113    17
           ..
usr_069     6
usr_048     6
usr_040     6
usr_012     5
usr_031     4
Name: count, Length: 120, dtype: int64

По выводу ясно, что каждый пользователь встречается один или более раз. Теперь посмотрим, как меняются данные у одного из пользователей.

In [36]:
df[df["user_id"] == "usr_012"]

,event_id,timestamp,user_id,department,source_ip,country,device_type,os_family,browser_family,auth_method,...,login_velocity_1h,distance_from_usual_km,account_age_days,password_age_days,known_device,new_country,vpn_used,analyst_verdict,analyst_risk_score,is_suspicious
70,evt_000693,2026-01-29 16:05:52,usr_012,HR,192.0.2.166,DE,tablet,iOS,Safari,password,...,2,62.0,2271,132,1,0,0,benign,22,0
699,evt_000151,2026-02-21 12:10:53,usr_012,HR,203.0.113.88,DE,tablet,iOS,Safari,password,...,3,0.4,2294,155,1,0,0,benign,51,0
716,evt_000442,2026-02-15 11:46:18,usr_012,HR,203.0.113.88,DE,tablet,iOS,Safari,password,...,4,NaN,2288,149,1,0,0,benign,60,0
940,evt_000166,2026-03-05 17:43:01,usr_012,HR,203.0.113.88,DE,tablet,iOS,Safari,password,...,2,19.1,2306,167,1,0,0,benign,3,0
1302,evt_000387,2026-01-22 15:57:07,usr_012,HR,203.0.113.88,DE,tablet,iOS,Safari,password,...,3,4.8,2264,125,1,0,0,benign,61,0


Как можно заметить, для usr_012 меняются некоторые данные. Разное время входа в систему, а также различные IP-адреса и т.д.

По полученным данным можно сделать вывод, что одна строка - это отдельное событие аутентификации, а не пользователя целиком.

## 1.2. Целевая переменная

is_suspicious - целевая переменная, т.е. она является результатом, который мы хотим предсказывать. По остальным признакам X будет определяться значение is_suspicious.

Посмотрим какие классы имеются у is_suspicious:

In [37]:
df["is_suspicious"].value_counts()

is_suspicious
0    1301
1     129
Name: count, dtype: int64

У is_suspicious два класса: 1 и 0. Из исходных данных известно, что 1 соответсвтует подозрительному событию, 0 нормальному событию.

## 1.3. Паспорт датасета

**Происхождение:** учебный синтетический набор данных, предоставленный для выполнения лабораторной работы.

**Дата получения:** 25.09.2026

**Исходный размер:** 1430 строк x 23 столбца

In [38]:
df.shape

(1430, 23)

**Ограничения:** так как данные являются синтетическими (искусственными), они не отражают полностью особенности реальных событий информационной безопасности. Распределение признаков и сценарии атак могут отличаться от реальных данных, поэтому результаты анализа нельзя переносить на настоящую инфраструктуру.

### 1.4. Таблица признаков

Посморим фактические типы:

In [39]:
df.dtypes

event_id                      str
timestamp                     str
user_id                       str
department                    str
source_ip                     str
country                       str
device_type                   str
os_family                     str
browser_family                str
auth_method                   str
hour                        int64
is_weekend                  int64
failed_attempts_24h         int64
login_velocity_1h           int64
distance_from_usual_km    float64
account_age_days            int64
password_age_days          object
known_device                int64
new_country                 int64
vpn_used                    int64
analyst_verdict               str
analyst_risk_score          int64
is_suspicious               int64
dtype: object

Составим таблицу:

| Название столбца         | Смысл                                              | Фактический тип | Ожидаемый тип            | Допустимые значения              | Момент доступности           | Решение об использовании         |
| ------------------------ | -------------------------------------------------- | --------------- | ------------------------ | -------------------------------- | ---------------------------- | -------------------------------- |
| `event_id`               | Уникальный идентификатор события                   | `str`           | строковый                | уникальные ID                    | В момент события             | Не использовать                  |
| `timestamp`              | Дата и время события                               | `str`           | datetime                 | корректные дата и время          | В момент события             | Использовать                     |
| `user_id`                | Идентификатор пользователя                         | `str`           | категориальный           | ID пользователей                 | В момент события               | Не использовать как X/Использовать для группировки при разбиении данных                     |
| `department`             | Подразделение пользователя                         | `str`           | категориальный           | значения подразделений           | В момент события            | Использовать                     |
| `source_ip`              | IP-адрес источника входа                           | `str`           | строковый | корректные IP-адреса             | В момент события               | Использовать                     |
| `country`                | Страна, связанная с попыткой входа                 | `str`           | категориальный           | названия стран                   | В момент события               | Использовать                     |
| `device_type`            | Тип устройства                                     | `str`           | категориальный           | значения типов устройств         | В момент события               | Использовать                     |
| `os_family`              | Семейство операционной системы                     | `str`           | категориальный           | значения ОС                      | В момент события               | Использовать                     |
| `browser_family`         | Семейство браузера                                 | `str`           | категориальный           | значения браузеров               | В момент события               | Использовать                     |
| `auth_method`            | Метод аутентификации                               | `str`           | категориальный           | допустимые методы аутентификации | В момент события               | Использовать                     |
| `hour`                   | Час события                                        | `int64`         | целое число              | 0-23                             | В момент события             | Использовать                     |
| `is_weekend`             | Признак выходного дня                              | `int64`         | бинарный                 | 0,1                             | В момент события             | Использовать                     |
| `failed_attempts_24h`    | Количество неудачных попыток за последние 24 часа  | `int64`         | целое число              | ≥ 0                              | В момент события           | Использовать                     |
| `login_velocity_1h`      | Интенсивность входов за последний час              | `int64`         | целое число              | ≥ 0                              | В момент события           | Использовать                     |
| `distance_from_usual_km` | Расстояние от обычного места входа                 | `float64`       | вещественное число       | ≥ 0                              | В момент события               | Использовать                     |
| `account_age_days`       | Возраст учётной записи в днях                      | `int64`         | целое число              | ≥ 0                              | До события                     | Использовать|
| `password_age_days`      | Возраст пароля в днях                              | `object`        | целое число              | ≥ 0                              | До события                     | Использовать |
| `known_device`           | Признак известного устройства                      | `int64`         | бинарный                 | 0,1                             | В момент события               | Использовать                     |
| `new_country`            | Признак новой страны для пользователя              | `int64`         | бинарный                 | 0,1                             | В момент события               | Использовать                     |
| `vpn_used`               | Признак использования VPN                          | `int64`         | бинарный                 | 0,1                             | В момент события               | Использовать                     |
| `analyst_verdict`        | Решение аналитика по событию                       | `str`           | категориальный           | значения вердиктов               | После анализа события        | Не использовать                  |
| `analyst_risk_score`     | Оценка риска аналитиком                            | `int64`         | целое число              | допустимый диапазон оценки       | После анализа события        | Не использовать                  |
| `is_suspicious`          | Целевая переменная: подозрительное событие или нет | `int64`         | бинарный                 | 0,1                             | После определения результата | **y, не использовать как X**     |

analyst_verdict и analyst_risk_score **являются результатами анализа событий** и поэтому не могут использоваться для обучения. Эти данные дают модели ответ, чем фактически является события.

# 2. Инвентаризация

Загрузка листа "Данные" файла .xlsx была выполнена в пункте 1.

## 2.1. Анализ структуры и размера датасета

Выведем первые строки датасета:

In [40]:
df.head()

,event_id,timestamp,user_id,department,source_ip,country,device_type,os_family,browser_family,auth_method,...,login_velocity_1h,distance_from_usual_km,account_age_days,password_age_days,known_device,new_country,vpn_used,analyst_verdict,analyst_risk_score,is_suspicious
0,evt_001370,2026-03-12 20:27:57,usr_046,Operations,203.0.113.178,SE,mobile,Android,Chrome,password,...,1,39.4,871,229,0,0,0,benign,65,0
1,evt_000493,2026-01-06 00:17:20,usr_068,Sales,203.0.113.132,RU,mobile,Android,Chrome,password,...,5,16.9,2383,11,1,0,0,benign,43,0
2,evt_000433,2026-01-27 17:57:11,usr_033,SOC,192.0.2.185,PL,laptop,Windows,Chrome,password,...,5,28.8,795,258,1,0,0,benign,50,0
3,evt_001361,2026-03-10 16:30:25,usr_085,Sales,192.0.2.91,NO,mobile,Android,Chrome,otp,...,3,26.1,1186,184,1,0,0,benign,40,0
4,evt_000791,2026-02-10 05:26:56,usr_092,R&D,203.0.113.190,DE,laptop,Windows,Chrome,fido2,...,6,86.0,784,182,1,0,0,benign,7,0


Выведем последние строки датасета:

In [41]:
df.tail()

,event_id,timestamp,user_id,department,source_ip,country,device_type,os_family,browser_family,auth_method,...,login_velocity_1h,distance_from_usual_km,account_age_days,password_age_days,known_device,new_country,vpn_used,analyst_verdict,analyst_risk_score,is_suspicious
1425,evt_000907,2026-03-09 00:01:20,usr_065,Operations,203.0.113.93,PL,tablet,iOS,Safari,push,...,2,49.8,3245,79,1,0,0,benign,15,0
1426,evt_001094,2026-01-17 09:15:30,usr_051,HR,192.0.2.165,RU,tablet,iOS,Safari,password,...,2,64.2,1803,239,1,0,0,benign,46,0
1427,evt_001253,2026-01-25 00:38:56,usr_061,R&D,198.51.100.128,RU,desktop,Windows,Chrome,otp,...,13,1088.3,2917,177,1,1,0,confirmed_threat,78,1
1428,evt_000743,2026-02-23 22:19:53,usr_043,HR,192.0.2.51,US,laptop,Windows,Edge,otp,...,5,14.4,1495,257,1,0,0,benign,31,0
1429,evt_000581,2026-02-14 19:29:43,usr_019,Finance,198.51.100.3,NL,laptop,Windows,Chrome,password,...,2,73.4,2915,unknown,1,0,0,benign,24,0


Посмотрим размер таблицы:

In [42]:
df.shape

(1430, 23)

Посмотрим названия столбцов:

In [43]:
df.columns

Index(['event_id', 'timestamp', 'user_id', 'department', 'source_ip',
       'country', 'device_type', 'os_family', 'browser_family', 'auth_method',
       'hour', 'is_weekend', 'failed_attempts_24h', 'login_velocity_1h',
       'distance_from_usual_km', 'account_age_days', 'password_age_days',
       'known_device', 'new_country', 'vpn_used', 'analyst_verdict',
       'analyst_risk_score', 'is_suspicious'],
      dtype='str')

Посмотрим типы данных и объём памяти:

In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1430 entries, 0 to 1429
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   event_id                1430 non-null   str    
 1   timestamp               1430 non-null   str    
 2   user_id                 1430 non-null   str    
 3   department              1416 non-null   str    
 4   source_ip               1430 non-null   str    
 5   country                 1430 non-null   str    
 6   device_type             1430 non-null   str    
 7   os_family               1430 non-null   str    
 8   browser_family          1400 non-null   str    
 9   auth_method             1430 non-null   str    
 10  hour                    1430 non-null   int64  
 11  is_weekend              1430 non-null   int64  
 12  failed_attempts_24h     1430 non-null   int64  
 13  login_velocity_1h       1430 non-null   int64  
 14  distance_from_usual_km  1398 non-null   float64
 15

## 2.2. Проверка объектов и целевого столбца

Проверим, что число строк соответсвует числу объектов:

In [45]:
df.shape[0]

1430

Полученное число строк соответствует числу объектов, так как каждая строка датасета представляет одно событие.

Проверим, что целевой столбец присутствует один раз:

In [46]:
list(df.columns).count("is_suspicious")

1

## 2.3. Описательная статистика признаков

Для числовых признаков выведем описательные статистики:

In [47]:
df.describe()

,hour,is_weekend,failed_attempts_24h,login_velocity_1h,distance_from_usual_km,account_age_days,known_device,new_country,vpn_used,analyst_risk_score,is_suspicious
count,1430.000000,1430.000000,1430.000000,1430.000000,1398.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000
mean,11.731469,0.280420,1.897902,3.197902,376.536409,1637.660140,0.911189,0.044755,0.118182,40.848951,0.090210
std,7.113637,0.449361,37.336751,2.222254,1853.187377,921.062124,0.284571,0.206838,0.322936,23.968772,0.286582
min,0.000000,0.000000,-1.000000,1.000000,-25.000000,-7.000000,0.000000,0.000000,0.000000,3.000000,0.000000
25%,6.000000,0.000000,0.000000,2.000000,15.700000,839.750000,1.000000,0.000000,0.000000,20.000000,0.000000
50%,12.000000,0.000000,0.000000,3.000000,31.350000,1656.500000,1.000000,0.000000,0.000000,40.000000,0.000000
75%,18.000000,1.000000,1.000000,4.000000,56.275000,2365.500000,1.000000,0.000000,0.000000,59.000000,0.000000
max,23.000000,1.000000,999.000000,18.000000,20000.000000,3266.000000,1.000000,1.000000,1.000000,99.000000,1.000000


Для категориальных признаков выведем число уникальных значений и наиболее частые категории:

In [48]:
categorical_columns = df.select_dtypes(include="str").columns # выбираем столбцы со строковым типом данных

for col in categorical_columns:
    print(f"\n- {col} -")
    print("Unique values:", df[col].nunique()) # кол-во уникальных значений
    print("Most frequent values:")
    print(df[col].value_counts().head())


- event_id -
Unique values: 1412
Most frequent values:
event_id
evt_000433    2
evt_000847    2
evt_001063    2
evt_001315    2
evt_000107    2
Name: count, dtype: int64

- timestamp -
Unique values: 1412
Most frequent values:
timestamp
2026-01-27 17:57:11    2
2026-02-16 18:53:44    2
2026-01-23 04:25:22    2
2026-02-21 02:47:56    2
2026-01-08 01:00:54    2
Name: count, dtype: int64

- user_id -
Unique values: 120
Most frequent values:
user_id
usr_062    24
usr_025    21
usr_082    19
usr_060    17
usr_113    17
Name: count, dtype: int64

- department -
Unique values: 8
Most frequent values:
department
SOC           269
R&D           248
HR            196
Operations    179
IT            176
Name: count, dtype: int64

- source_ip -
Unique values: 320
Most frequent values:
source_ip
198.51.100.81    17
192.0.2.28       14
192.0.2.242      14
192.0.2.95       13
198.51.100.60    13
Name: count, dtype: int64

- country -
Unique values: 9
Most frequent values:
country
NO    393
DE    268

Сохраним исходное число строк и столбцов в переменные:

In [49]:
orig_rows, orig_cols = df.shape
print(orig_rows, orig_cols)

1430 23


# 3. Проверка типов и допустимых значений

## 3.1. Проверка типов и форматов данных

Выведем фактические типы каждого столбца:

In [50]:
df.dtypes

event_id                      str
timestamp                     str
user_id                       str
department                    str
source_ip                     str
country                       str
device_type                   str
os_family                     str
browser_family                str
auth_method                   str
hour                        int64
is_weekend                  int64
failed_attempts_24h         int64
login_velocity_1h           int64
distance_from_usual_km    float64
account_age_days            int64
password_age_days          object
known_device                int64
new_country                 int64
vpn_used                    int64
analyst_verdict               str
analyst_risk_score          int64
is_suspicious               int64
dtype: object

Основываясь на таблице из пункта 1.4 можно сделать вывод, что фактический тип столбцов password_age_days и timestamp не соответствует их смыслу:
- password_age_days в результате имеет тип object, но столбец соответствует количеству дней и потому должен быть тип int;
- timestamp имеет тип str, но ожидаемый тип datetime.

Посмотрим, почему для password_age_days тип определяется как object:

In [51]:
df["password_age_days"].unique()

array([229, 11, 258, 184, 182, 114, 211, 95, 103, 328, 319, 324, 291, 272,
       206, 87, 298, 256, 73, 198, 281, 343, 380, 242, 239, 235, 335, 267,
       107, 117, 219, 332, 168, 310, 204, 176, 193, 270, 260, 263, 118,
       334, 162, 143, 311, 352, 308, 269, 216, 398, 317, 275, 222, 158,
       94, 227, 364, 188, 252, 224, 234, 359, 132, 90, 257, 106, 301, 152,
       104, 189, 215, 326, 101, 194, 246, 323, 146, 147, 331, 283, 296,
       341, 49, 315, 322, 191, 299, 226, 231, 240, 71, 273, 141, 91, 100,
       284, 247, 126, 18, 383, 261, 228, 254, 115, 236, 321, 243, 401,
       309, 294, 83, 318, 274, 277, 199, 109, 185, 119, 233, 213, 369, 75,
       336, 177, 238, 202, 253, 248, 165, 85, 164, 180, 135, 183, 137, 19,
       265, 61, 192, 244, 133, 230, 'unknown', 69, 196, 339, 249, 327, 79,
       89, 342, 178, 289, 330, 320, 279, 208, 150, 130, 360, 286, 218,
       124, 313, 98, 108, 163, 41, 212, 149, 292, 99, 393, 278, 70, 223,
       285, 302, 355, 266, 237, 145, 316, 306

В списке можно заметить строковое значение 'unknown', по этой причине для password_age_days не определялся тип int.

Теперь найдем смешанные форматы дат. Для этого сначала посмотрим формат, который встречается в датасете:

In [52]:
df["timestamp"]

0       2026-03-12 20:27:57
1       2026-01-06 00:17:20
2       2026-01-27 17:57:11
3       2026-03-10 16:30:25
4       2026-02-10 05:26:56
               ...         
1425    2026-03-09 00:01:20
1426    2026-01-17 09:15:30
1427    2026-01-25 00:38:56
1428    2026-02-23 22:19:53
1429    2026-02-14 19:29:43
Name: timestamp, Length: 1430, dtype: str

В результате преобладает формат 'YYYY-MM-DD HH:MM:SS', поэтому его возьмем за основу.

Напишем программу, которая выведет неподходящие под шаблон форматы, а также даты, которые не существуют.

In [53]:
parsed = pd.to_datetime( # преобразуем данные по заданному формату
    df["timestamp"],
    format="%Y-%m-%d %H:%M:%S", # заданный шаблон
    errors="coerce" # если формат не подходит, то ставим NaT
)

display( # показать полученный результат в виде таблицы
    df.loc[ # выбираем конкретные строки и столбцы из таблицы df
        parsed.isna(), # выбираем только те строки где стоит NaT
        ["event_id", "timestamp"] # отображаемые столбцы
        ]
)

,event_id,timestamp
418,evt_000920,not_recorded
422,evt_001299,17.03.2026 13:45
727,evt_001047,2026/03/12 25:61
1034,evt_000098,2026-02-30 09:15:00


Получилось 4 результата: значение 'not_recorded', 1 отличный от основного формата, несуществующее время 25:61 и 30 февраля.

Далее найдем категориальные признаки с ошибочными вариантами написания:

In [54]:
categorical_columns = [
    "department",
    "country",
    "device_type",
    "os_family",
    "browser_family",
    "auth_method",
    "analyst_verdict"
]

for col in categorical_columns:
    print("\n")
    print(df[col].value_counts())



department
SOC           269
R&D           248
HR            196
Operations    179
IT            176
Sales         174
Finance       166
unknown         8
Name: count, dtype: int64


country
NO    393
DE    268
SE    257
NL    188
US    135
PL     95
RU     80
BR      8
CN      6
Name: count, dtype: int64


device_type
mobile     491
tablet     334
laptop     329
desktop    276
Name: count, dtype: int64


os_family
Android    464
Windows    409
iOS        361
macOS      123
Linux       73
Name: count, dtype: int64


browser_family
Chrome     793
Safari     263
Edge       182
Firefox    152
-           10
Name: count, dtype: int64


auth_method
password    571
push        372
otp         330
fido2       157
Name: count, dtype: int64


analyst_verdict
benign              1301
confirmed_threat     129
Name: count, dtype: int64


По результату можно сделать вывод, что категориальных признаков с ошибочными вариантами написания не было найдено.

## 3.2. Предметные ограничения

Необходимо сформулировать предметные ограничения для счетчиков, расстояний, возраста учётной записи и бинарных признаков.

В таблице из пункта 1.4 прописаны допустимые значения. Проверим, нарушают ли какие то строки эти ограничения.

Под проверку попадают следующие столбцы:
- failed_attempts_24h
- login_velocity_1h
- distance_from_usual_km
- account_age_days
- password_age_days
- hour
- is_weekend
- known_device
- new_country
- vpn_used
- is_suspicious

Проверим для hour:

In [55]:
df.loc[~df["hour"].between(0, 23), ["event_id", "hour"]]

,event_id,hour


Все значения в допустимом диапазоне.

Проверим для бинарных признаков:

In [56]:
columns = [
    "is_weekend",
    "known_device",
    "new_country",
    "vpn_used",
    "is_suspicious"
]

for col in columns:
    print("\n")
    display(df.loc[~df[col].isin([0, 1]), ["event_id", col]])

,event_id,is_weekend


,event_id,known_device


,event_id,new_country


,event_id,vpn_used


,event_id,is_suspicious


По результату видно, что все значения в допустимом диапазоне.

Проверим счётчики, расстояние и возраст аккаунта:

In [57]:
cols = ["failed_attempts_24h", 
        "login_velocity_1h", 
        "distance_from_usual_km", 
        "account_age_days"
]

for col in cols:
    print("\n")
    display(df.loc[df[col] < 0, ["event_id", col]])

,event_id,failed_attempts_24h
407,evt_000829,-1
877,evt_000066,-1
1106,evt_000781,-1


,event_id,login_velocity_1h


,event_id,distance_from_usual_km
379,evt_000186,-25.0
480,evt_000594,-25.0
1027,evt_000804,-25.0
1192,evt_000440,-25.0


,event_id,account_age_days
436,evt_000067,-7
843,evt_000024,-7
1171,evt_000925,-7


По результату видно, что были найдены технические ошибки. Отрицательные значения невозможны для данных признаков, поэтому их необходимо исправить.

Далее проверим, что расстояние не превышает максимальное возможное (20004 км):

In [58]:
display(df.loc[df["distance_from_usual_km"] > 20004, ["event_id", "distance_from_usual_km"]])

,event_id,distance_from_usual_km


Такие строки не нашлись.

Также проверим, существуют ли строки, где возраст пароля больше возраста аккаунта:

In [59]:
password_age = pd.to_numeric(
    df["password_age_days"].replace("unknown", np.nan),
    errors="coerce"
)

mask = password_age > df["account_age_days"]

print("Total:", mask.sum())

Total: 93


Были найдены 93 строки. Это невозможно и потому подлежит замене.

## 3.3. Исправление

Начнем с исправления отрицательных значений. Так как мы точно не знаем каким образом они получилось отрицательными, то заменим значения на NaN:

In [60]:
cols = ["failed_attempts_24h", 
        "distance_from_usual_km", 
        "account_age_days"
]

k = 0

for col in cols:
    k += int((df[col] < 0).sum())
    print(f"{col}: {(df[col] < 0).sum()} values replaced")
    df.loc[df[col] < 0, col] = np.nan

print(f"Total: {k}")

failed_attempts_24h: 3 values replaced
distance_from_usual_km: 4 values replaced
account_age_days: 3 values replaced
Total: 10


Как видно по результату, всего в сумме было выполнено 10 замен.

Далее выполним исправления для дат. Из пункта 3.1 известно, что есть 3 некорректные строки и одна строка с отличным от основного форматом.

Исправим 3 некорректные строки на NaT и изменим формат строки c другим форматом:

In [61]:
# Преобразуем даты стандартного формата
parsed = pd.to_datetime(
    df["timestamp"],
    format="%Y-%m-%d %H:%M:%S", # шаблон стандартного формата
    errors="coerce" # если дата не подходит, то записываем NaT вместо ошибки
)

# Находим даты, записанные в другом формате
mask = df["timestamp"].str.match( # проверяем, соответствует ли текст заданному шаблону
    r"^\d{2}\.\d{2}\.\d{4} \d{2}:\d{2}$", # r".." - правило поиска текста. Отличный от основного шаблон: DD.MM.YYYY HH:MM
    na=False # если значение пустое, значит оно не соответствует шаблону
)

# Меняем строку с отличным форматом от основного
parsed.loc[mask] = pd.to_datetime( # записать результат в переменную parsed, но только в те строки, которые отмечены mask
    df.loc[mask, "timestamp"],
    format="%d.%m.%Y %H:%M",
    errors="coerce" # NaT вместо ошибки
)

df["timestamp"] = parsed

print("NaT count:", df["timestamp"].isna().sum())

# Cтроки с NaT
display(
    df.loc[
        df["timestamp"].isna(),
        ["event_id", "timestamp"]
    ]
)

NaT count: 3


,event_id,timestamp
418,evt_000920,NaT
727,evt_001047,NaT
1034,evt_000098,NaT


Таким образом, были исправлены 3 некорректные строки и изменен формат строки с другим форматом.

Заменим значения возраста пароля, где он превышает возраст аккаунта на NaN:

In [62]:
password_age = pd.to_numeric(
    df["password_age_days"].replace("unknown", np.nan), # временно избавимся от unknown
    errors="coerce"
)

mask = password_age > account_age

df.loc[mask, "password_age_days"] = np.nan

print(f"Total: {mask.sum()}")

Total: 93


Заменены 93 строки.

# 4. Анализ пропусков

## 4.1. Расчёт числа и доли пропусков

Рассчитаем число и долю пропусков для каждого столбца:

In [63]:
# Считаем количество пропусков в каждом столбце
missing_count = df.isna().sum()

# Считаем долю пропусков в процентах
missing_percent = df.isna().mean() * 100

# Собираем результаты в таблицу
missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

# Оставляем только столбцы с пропусками
missing_table = missing_table[missing_table["missing_count"] > 0]

# Сортируем по убыванию доли пропусков
missing_table = missing_table.sort_values(
    by="missing_percent",
    ascending=False
)

display(missing_table.round(2))


,missing_count,missing_percent
password_age_days,93,6.50
distance_from_usual_km,36,2.52
browser_family,30,2.10
department,14,0.98
timestamp,3,0.21
failed_attempts_24h,3,0.21
account_age_days,3,0.21


## 4.2. Скрытые значения

Найдем скрытые значения:

In [64]:
# Возможные обозначения пропусков
markers = [
    "?", "unknown", "-", "not_recorded",
    "n/a", "null", "none", "missing",
    "not available", "", "0"
]

# Проверяем текстовые столбцы
for col in df.select_dtypes(include=["object", "str"]).columns:
    values = df[col].astype("string").str.strip().str.lower()

    for marker in markers:
        count = (values == marker).sum()

        if count > 0:
            print(f"{col}: '{marker}' — {count} values")

department: 'unknown' — 8 values
browser_family: '-' — 10 values
password_age_days: 'unknown' — 9 values


Нашлись 3 признака со скрытыми пропусками.

## 4.3. Решение проблемных признаков

В итоге получился следующий список признаков:
- distance_from_usual_km
- browser_family
- department
- timestamp
- failed_attempts_24h
- account_age_days
- password_age_days

Первым делом преобразуем скрытые значения 'unknown' и '-' в настоящие пропуски:

In [65]:
# Убираем пробелы и приводим текст к нижнему регистру
df["department"] = df["department"].apply(
    lambda x: x.strip().lower() if isinstance(x, str) else x
)

df["password_age_days"] = df["password_age_days"].apply(
    lambda x: x.strip().lower() if isinstance(x, str) else x
)

# Заменяем маркеры пропусков
df["browser_family"] = df["browser_family"].replace(
    r"^\s*-\s*$", np.nan, regex=True
)

df["department"] = df["department"].replace("unknown", np.nan)

df["password_age_days"] = df["password_age_days"].replace(
    "unknown", np.nan
)

# Преобразуем возраст пароля в число
df["password_age_days"] = pd.to_numeric(
    df["password_age_days"], errors="coerce"
)

print(df[[
    "browser_family",
    "department",
    "password_age_days"
]].isna().sum())

browser_family        40
department            22
password_age_days    102
dtype: int64


Для начала примем решение для timestamp. Посмотрим сколько всего есть строк с NaT:

In [67]:
print("NaT count:", df["timestamp"].isna().sum())
print("Total lines:", len(df))

NaT count: 3
Total lines: 1430


Количество строк с NaT составляет 3 из 1430. В таком случае приемлемым решением является удаление этих строк:

In [69]:
df = df.dropna(subset=["timestamp"]).copy()

print("NaT count:", df["timestamp"].isna().sum())
print("Total lines:", len(df))

NaT count: 0
Total lines: 1427


Некорректные значения в timestamp удалены, поскольку время попытки входа важно для определения часа и даты события, а восстановить правильное время по имеющимся данным невозможно. Таких строк всего 3 (0,21% датасета), поэтому их удаление позволяет избежать использования недостоверных временных данных с минимальной потерей информации.

Пропуски в категориальных признаках на этапе подготовки данных оставлены без заполнения. В дальнейшем внутри pipeline они будут заменены на отдельную категорию Unknown. Это позволит сохранить информацию об отсутствии данных и избежать ошибочного присвоения наиболее частой категории.

Для числовых признаков failed_attempts_24h, account_age_days, password_age_days и distance_from_usual_km пропуски будут заполнены медианой внутри pipeline. Пропуски заполняются медианой, поскольку она позволяет сохранить строки с полезной информацией. Медиана отражает типичное значение признака и подходит для заполнения пропусков в таких показателях.